# Resultados Consolidados - RAG Evaluation

Comparação entre todos os modelos testados com `gpt-4o-mini` como avaliador.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 150

GRAFICOS_DIR = 'graficos_tcc'
os.makedirs(GRAFICOS_DIR, exist_ok=True)

In [ ]:
csv_files = sorted(glob.glob('result_gpt4omini_*.csv'))
csv_files

In [ ]:
# Classificação confirmada em metrics.ipynb: NoiseSensitivity, harmfulness e
# maliciousness são métricas em que valores menores indicam melhor desempenho.
metric_columns = [
    'context_precision', 'context_recall', 'context_entity_recall',
    'noise_sensitivity_relevant', 'answer_relevancy', 'faithfulness',
    'factual_correctness', 'semantic_similarity', 'non_llm_string_similarity',
    'rouge_score', 'string_present', 'exact_match',
    'harmfulness', 'maliciousness', 'coherence', 'correctness', 'conciseness'
]

metricas_menor_melhor = [
    'noise_sensitivity_relevant',  # nome da métrica RAGAS NoiseSensitivity
    'harmfulness',
    'maliciousness',
]
metricas_maior_melhor = [m for m in metric_columns if m not in metricas_menor_melhor]

assert len(metricas_maior_melhor) == 14
assert len(metricas_menor_melhor) == 3
assert set(metricas_maior_melhor) | set(metricas_menor_melhor) == set(metric_columns)
assert not set(metricas_maior_melhor) & set(metricas_menor_melhor)

results = {}
for f in csv_files:
    model_name = f.replace('result_gpt4omini_', '').replace('.csv', '')
    df = pd.read_csv(f)
    available = [c for c in metric_columns if c in df.columns]
    means = df[available].mean(numeric_only=True)
    results[model_name] = means

df_means = pd.DataFrame(results).T
df_means.index.name = 'modelo'
df_means = df_means.round(4)
df_means['media_maior_melhor'] = df_means[metricas_maior_melhor].mean(axis=1)
df_means['media_menor_melhor'] = df_means[metricas_menor_melhor].mean(axis=1)
df_means

## Gráfico Comparativo - Todas as Métricas

In [ ]:
df_plot = df_means.copy()

fig, axes = plt.subplots(6, 3, figsize=(18, 24))
axes = axes.flatten()

for i, metric in enumerate(df_plot.columns):
    ax = axes[i]
    menor_melhor = metric in metricas_menor_melhor
    sorted_data = df_plot[metric].sort_values(ascending=menor_melhor)
    melhor = sorted_data.min() if menor_melhor else sorted_data.max()
    pior = sorted_data.max() if menor_melhor else sorted_data.min()
    colors = ['#2ecc71' if v == melhor else '#e74c3c' if v == pior else '#3498db' for v in sorted_data]
    bars = ax.barh(range(len(sorted_data)), sorted_data.values, color=colors, edgecolor='white', height=0.6)
    ax.set_yticks(range(len(sorted_data)))
    ax.set_yticklabels(sorted_data.index, fontsize=8)
    ax.set_xlim(0, 1.05)
    sentido = 'menor = melhor' if menor_melhor else 'maior = melhor'
    ax.set_title(f'{metric} ({sentido})', fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', labelsize=7)
    for bar, val in zip(bars, sorted_data.values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
                va='center', fontsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Comparação de Métricas por Modelo', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, 'metricas_individual.png'), bbox_inches='tight', dpi=200)
plt.show()

## Heatmap Comparativo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8), gridspec_kw={'width_ratios': [14, 3]})

sns.heatmap(df_means[metricas_maior_melhor], annot=True, fmt='.3f', cmap='RdYlGn',
            linewidths=0.5, cbar_kws={'label': 'Score (maior = melhor)'},
            vmin=0, vmax=1, ax=axes[0])
axes[0].set_title('Métricas em que maior é melhor', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].tick_params(axis='y', rotation=0, labelsize=9)

sns.heatmap(df_means[metricas_menor_melhor], annot=True, fmt='.3f', cmap='RdYlGn_r',
            linewidths=0.5, cbar_kws={'label': 'Score (menor = melhor)'},
            vmin=0, vmax=1, ax=axes[1])
axes[1].set_title('Métricas em que menor é melhor', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=8)
axes[1].tick_params(axis='y', left=False, labelleft=False)

fig.suptitle('Heatmap Comparativo das Métricas por Modelo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, 'heatmap_comparativo.png'), bbox_inches='tight', dpi=200)
plt.show()

## Médias Separadas por Sentido da Métrica

A média bruta das 17 métricas não é usada como ranking principal, pois combina métricas de sentidos opostos. Para rastreabilidade, o CSV preserva as 17 métricas individuais e acrescenta as duas médias abaixo: 14 métricas em que maior é melhor e 3 métricas em que menor é melhor.

In [ ]:
ranking_maior = df_means['media_maior_melhor'].sort_values(ascending=False)
ranking_menor = df_means['media_menor_melhor'].sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, ranking, titulo, ylabel in [
    (axes[0], ranking_maior, '14 métricas: maior é melhor', 'Média das métricas'),
    (axes[1], ranking_menor, '3 métricas: menor é melhor', 'Média das métricas'),
]:
    colors = ['#2ecc71' if i == 0 else '#e74c3c' if i == len(ranking)-1 else '#3498db' for i in range(len(ranking))]
    bars = ax.bar(ranking.index, ranking.values, color=colors, edgecolor='white', width=0.5)
    for bar, val in zip(bars, ranking.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=30, labelsize=9)

fig.suptitle('Médias por Sentido das Métricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, 'ranking_geral.png'), bbox_inches='tight', dpi=200)
plt.show()

for titulo, ranking in [('Maior é melhor', ranking_maior), ('Menor é melhor', ranking_menor)]:
    print(f'\n{titulo}:')
    for i, (model, score) in enumerate(ranking.items(), 1):
        print(f'{i:2}. {model:35s} {score:.4f}')

## Análise por Categoria

- **Recuperação**: context_precision, context_recall, context_entity_recall
- **Sensibilidade ao ruído**: noise_sensitivity_relevant *(menor = melhor)*
- **Qualidade da Resposta**: answer_relevancy, faithfulness, factual_correctness
- **Similaridade**: semantic_similarity, non_llm_string_similarity, rouge_score
- **Correspondência lexical estrita**: string_present, exact_match
- **Segurança**: harmfulness *(menor = melhor)*, maliciousness *(menor = melhor)*
- **Estilo**: coherence, correctness, conciseness

In [ ]:
categorias = {
    'Recuperação': ['context_precision', 'context_recall', 'context_entity_recall'],
    'Sensibilidade ao ruído': ['noise_sensitivity_relevant'],
    'Qualidade': ['answer_relevancy', 'faithfulness', 'factual_correctness'],
    'Similaridade': ['semantic_similarity', 'non_llm_string_similarity', 'rouge_score'],
    'Correspondência lexical estrita': ['string_present', 'exact_match'],
    'Segurança': ['harmfulness', 'maliciousness'],
    'Estilo': ['coherence', 'correctness', 'conciseness'],
}

fig, axes = plt.subplots(3, 3, figsize=(20, 16))
axes = axes.flatten()

for i, (cat_name, cat_metrics) in enumerate(categorias.items()):
    ax = axes[i]
    available = [m for m in cat_metrics if m in df_means.columns]
    menor_melhor = set(available).issubset(metricas_menor_melhor)
    assert menor_melhor or set(available).issubset(metricas_maior_melhor), f'Categoria mista: {cat_name}'
    cat_data = df_means[available].mean(axis=1).sort_values(ascending=menor_melhor)
    melhor = cat_data.iloc[0]
    pior = cat_data.iloc[-1]
    colors = ['#2ecc71' if v == melhor else '#e74c3c' if v == pior else '#3498db' for v in cat_data]
    bars = ax.barh(cat_data.index, cat_data.values, color=colors, edgecolor='white', height=0.5)
    ax.invert_yaxis()
    sentido = 'menor = melhor' if menor_melhor else 'maior = melhor'
    ax.set_xlabel(f'Média ({sentido})', fontsize=9)
    ax.set_title(f'{cat_name}', fontsize=12, fontweight='bold')
    ax.set_xlim(0, 1.05)
    ax.tick_params(axis='y', labelsize=8)
    for bar, val in zip(bars, cat_data.values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
                va='center', fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Média por Categoria de Métrica', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, 'categorias.png'), bbox_inches='tight', dpi=200)
plt.show()

## Tabela Completa

In [ ]:
df_means_sorted = df_means.sort_values('media_maior_melhor', ascending=False)
df_means_sorted

In [ ]:
df_means_sorted.to_csv('consolidated_means.csv')
print('Salvo: consolidated_means.csv')